In [1]:
import json
import os
annt_path='/home/v-jinjzhao/datasets/jierun/ref_msra_test_coco_o365.json'
with open(annt_path) as f:
    all_annts=json.load(f)
with open('/home/v-jinjzhao/dev/PSALM/output_jierun/pred.jsonl') as f:
    all_preds=[json.loads(line) for line in f]
print(len(all_annts),len(all_preds))

45341 45341


In [2]:
all_pred_bboxes=[]
all_gt_bboxes=[]
for annt,pred in zip(all_annts,all_preds):

    x,y,w,h=annt['bbox']
    gt_bbox=[x,y,x+w,y+h]
    pred_bboxes=pred['bbox'][0]
    all_gt_bboxes.append(gt_bbox)
    all_pred_bboxes.append(pred_bboxes)

In [3]:
import json
import torch
from tqdm import tqdm
from overlaps import bbox_overlaps
# import average from math
from statistics import mean


def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs




pred_bboxes=torch.tensor(all_pred_bboxes)
gt_bboxes=torch.tensor(all_gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.6725921351536137
thresh=0.7, acc=0.6056769810987848
thresh=0.9, acc=0.4411459826646964
iou|0.5:0.9=0.5882632593997583
tensor([0.8897, 0.0240, 0.8773,  ..., 0.0000, 0.0000, 0.0000])
{0.5: 0.6725921351536137, 0.55: 0.6572638450850223, 0.6: 0.6408548554288613, 0.65: 0.6236739374958646, 0.7: 0.6056769810987848, 0.75: 0.5822324165766084, 0.8: 0.5554575329172272, 0.85: 0.5154716481771465, 0.9: 0.4411459826646964}
45341/45341


In [4]:
for idx,annt in enumerate(all_annts):
    annt['PSALM_pred']=dict(
        pred_bboxes=pred_bboxes[idx].tolist(),
        iou=iou[idx].item()
    )


In [6]:
all_annts[9]

{'ann_id': 16978,
 'caption': "The giraffe, whose neck stretched high above the jeep's roof, was situated on the vehicle's right side and nearest to the camera.",
 'bbox': [253.89, 12.57, 307.06, 215.39],
 'bbox_area': 17566.607399999997,
 'bbox_id': 'coco_601155',
 'ori_category_id': 'coco_25',
 'image_id': 'coco_177353',
 'height': 485,
 'width': 640,
 'file_name': 'COCO_train2014_000000177353.jpg',
 'is_rewrite': 1,
 'shikra7b_pred': {'iou': 0.851250410079956,
  'pred_bbox': [259.8399963378906,
   2.3600006103515625,
   584.9599609375,
   232.760009765625]},
 'groundingGPT_pred': {'pred_bboxes': [249.60000610351562,
   6.199999809265137,
   550.4000244140625,
   243.0],
  'iou': 0.8688943386077881},
 'Lenna_pred': {'pred_bboxes': [98.15576171875,
   227.2060546875,
   316.17498779296875,
   467.6623840332031],
  'iou': 0.0003962365444749594},
 'PSALM_pred': {'pred_bboxes': [258, 15, 562, 228], 'iou': 0.9720191955566406}}

In [7]:
with open(annt_path,'w') as f:
    json.dump(all_annts,f,indent=4)